# K-Means & PCA — The Library Version

Both models via scikit-learn, verifying the scratch builds — plus the two-line pipeline (README §2.5) and the knobs you hand-rolled (`n_init` restarts, `k-means++`, `n_components`).

In [1]:
import pandas as pd
import numpy as np
from sklearn.preprocessing import StandardScaler
from sklearn.cluster import KMeans
from sklearn.decomposition import PCA
from sklearn.pipeline import make_pipeline

df = pd.read_csv("data/customer_data.csv")
X = df.values.astype(float)
Xs = StandardScaler().fit_transform(X)          # life-or-death, both models (README §4)

km = KMeans(n_clusters=4, n_init=10, init="k-means++", random_state=0).fit(Xs)
print(f"KMeans inertia: {km.inertia_:.1f}   <- same ballpark as the scratch build's best-of-restarts")
print(f"n_init=10 is our Block 9 restart insurance; 'k-means++' is the smarter opening (README §3.2)\n")

pca = PCA(n_components=2).fit(Xs)
print(f"PCA variance explained: {pca.explained_variance_ratio_.round(3)}  sum={pca.explained_variance_ratio_.sum():.0%}")
print("(matches the scratch eigen-decomposition — same eigenvectors, professionally packaged)")

KMeans inertia: 231.4   <- same ballpark as the scratch build's best-of-restarts
n_init=10 is our Block 9 restart insurance; 'k-means++' is the smarter opening (README §3.2)

PCA variance explained: [0.566 0.408]  sum=97%
(matches the scratch eigen-decomposition — same eigenvectors, professionally packaged)


In [2]:
# The payoff pipeline, industrial edition: two lines.
pipe = make_pipeline(StandardScaler(), PCA(n_components=2), KMeans(n_clusters=4, n_init=10, random_state=0))
segments = pipe.fit_predict(X)

prof = df.groupby(segments).mean().round(1)
prof.index = [f"segment {j}" for j in prof.index]
print("Segment profiles (the reading that turns geometry into meaning — README §5):")
print(prof)
print()
print("Same discipline as the scratch notebook: the pipeline hands you geometry;")
print("YOU validate the meaning before anyone calls these 'customer types' (README §8 Q6).")

Segment profiles (the reading that turns geometry into meaning — README §5):
           annual_income_k  spending_score  visits_per_month  avg_basket  \
segment 0            101.1            80.3              14.5        76.7   
segment 1             44.0            82.1              12.2        37.6   
segment 2             23.4            25.4               2.4        17.0   
segment 3             95.0            33.7               6.9        66.7   

           online_ratio   age  
segment 0           0.7  39.9  
segment 1           0.8  29.5  
segment 2           0.4  39.9  
segment 3           0.3  50.0  

Same discipline as the scratch notebook: the pipeline hands you geometry;
YOU validate the meaning before anyone calls these 'customer types' (README §8 Q6).


**The takeaway:** `KMeans.fit()` is Blocks 4–5 with k-means++ openings and `n_init` restarts built in; `PCA.fit()` is Block 7's five lines with numerical care; `make_pipeline` is Block 8. The judgment calls — K, components kept, and what the clusters *mean* — remain yours. No library ships those.